In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. **Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`**
   * 02-02. Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`
   * 02-03. Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`
   * 02-04. Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`
   * 02-05. Tools
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools
      * 02-05-02. Image Generation and Editing with `ImageGenerationTool`
      * 02-05-03. Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. Local MCP — SQLite Database: `MCPServerStdio`
      * 02-05-06. Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication
      * 02-05-07. AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. Local LLM via LiteLLM and Ollama: `LitellmModel`
      * 02-05-10. Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---
# Single Agent System — Python Tutor
* Simple agent that answers Python questions
* Demonstrates `Agent`, `Runner`
* `Runner` generally launches `Agent` asynchronously and code `await`s result
    * Can use `run_sync` to run synchronously
    * **Can't run synchronously in a notebook** — already has async event loop running
* For scripts: `asyncio.run(main())` created an asynchronous event loop 
* Agents are stateless by default 
    * No context maintained between runs
* [Traces dashboard](https://platform.openai.com/traces)     

In [ ]:
"""A simple single-agent system that answers Python questions."""
from agents import Agent, Runner, trace
from IPython.display import Markdown, display

# create an Agent
tutor = Agent(
    name="Python Tutor",
    model='gpt-5.5', # gpt-5.4-mini is the current default
    instructions="""You are a Python tutor for novice programmers. 
        Provide concise, simple answers to Python questions."""
)

In [ ]:
print('I am a Python tutor. How can I help you?') 
turn_count = 1 # turn counter for user-input prompts

# run until the user input is 'exit'
while ((prompt := input(f'\nInput [{turn_count}]: ')) != 'exit'):
    # launch agent asynchronously and await its response
    with trace(f'02-01-python-tutor-[{turn_count}]'):
        result = await Runner.run(tutor, prompt) 
    
    display(Markdown(result.final_output)) 
    #print(result.final_output) # show answer from tutor agent
    turn_count += 1 

print('User terminated app.')

## Sample Inputs
* What are the `print` function's parameters?
* Given a list containing 1-10, what's the easiest way to total them?
* Give me another way to do the same thing.
    * In a real conversation, context from prior question would factor into this answer

## Visualizing the Agent Workflow
[Open util.py](./source/util.py)

In [ ]:
from agents.extensions.visualization import draw_graph
draw_graph(tutor, filename="agent_workflow.png")

In [ ]:
from source.util import get_styled_agent_graph
get_styled_agent_graph(tutor)

## Learn More
* [Agents](https://openai.github.io/openai-agents-python/agents/)
* [Running agents](https://openai.github.io/openai-agents-python/running_agents/)
* [Results](https://openai.github.io/openai-agents-python/results/)

---
&copy; 2026 by Deitel & Associates, Inc. All Rights Reserved. 